### Logging

In [2]:
import logging

In [ ]:
# Configure a minimal logger
logging.basicConfig(level=logging.DEBUG)

# Get a logger named 'demo' — the name shows up in every log line
logger = logging.getLogger('demo')

# Log one message per level
logger.debug('Debug — detailed info')
logger.info('Info — normal event')
logger.warning('Warning — something odd')
logger.error('Error — something failed')
logger.critical('Critical — app cannot continue')

DEBUG:demo:Debug — detailed info
INFO:demo:Info — normal event
ERROR:demo:Error — something failed
CRITICAL:demo:Critical — app cannot continue


#### Handler

What is a Handler?
* A handler is the object that sends log messages somewhere.

A **handler** decides where log messages go:

**Handler**	Sends messages to:

* StreamHandler: Console (terminal)
* FileHandle: A file on disk
* NullHandler: Nowhere (silent)

In [ ]:
# Step 1: Reset the Logging System to avoid duplicate

# Get a copy of the current handlers
for h in logging.root.handlers[:]:
    
    # Remove each one
    logging.root.removeHandler(h)
    
# Step 2: Get the Root Logger and Set Its Level
root = logging.getLogger()

# Minimum level this logger accepts
root.setLevel(logging.DEBUG)

# Step 3: Create a handler that writes to the console
console = logging.StreamHandler()

# Step 4: Set the Handler's Level
console.setLevel(logging.DEBUG)

# Step 5: Create a Formatter
formatter = logging.Formatter('%(levelname)-8s | %(message)s')   # Format template

# Step 6: Attach the Formatter to the Handler
console.setFormatter(formatter)

# Step 7: Attach handler to the root logger
root.addHandler(console)

# Test
logger = logging.getLogger('demo')
logger.debug('Debug message')
logger.info('Info message')
logger.error('Error message')

DEBUG    | Debug message
INFO     | Info message
ERROR    | Error message


File Handler 
* File handler writes to a file

In [7]:
from pathlib import Path

# Folder where log files live
log_dir = Path(r'C:\Users\USER\rag_course\logs')
log_dir.mkdir(parents=True, exist_ok=True)

# The log file
log_file = log_dir / 'agrivoice.log'

print(f'Log file: {log_file}')

Log file: C:\Users\USER\rag_course\logs\agrivoice.log


In [8]:
# Step 2: Create a File Handler
file_handler = logging.FileHandler(log_file, encoding='utf-8')

# Step 3: Set the File Handler's Level
file_handler.setLevel(logging.WARNING)

# Step 4: Create a Formatter
formatter = logging.Formatter('%(levelname)-8s | %(message)s')   # Format template

# Step 5: Attach the Formatter
file_handler.setFormatter(formatter)

# Step 6: Attach to the Root Logger
root.addHandler(file_handler)

# Step 6: Test It
logger.info('This goes to console only')
logger.warning('This goes to console AND file')
logger.error('This also goes to console AND file')

INFO     | This goes to console only
WARNING  | This goes to console AND file
ERROR    | This also goes to console AND file


In [10]:
# Step 7: Read the File Back
print('File contents:')
print(log_file.read_text(encoding='utf-8'))

File contents:
WARNING  | This goes to console AND file
ERROR    | This also goes to console AND file



Scanner 

In [11]:
from pathlib import Path

# Where the file will be saved
output_path = Path(r'C:\Users\USER\rag_course\15_pure_python\agrivoice\services\file_scanner.py')

# The source code
source = '''"""
Simple file scanner for AgriVoice.

Scans a folder for images and logs every action.
Demonstrates pathlib + logging working together.
"""

import logging
from pathlib import Path

# Module-level logger — name shows up in every log line
logger = logging.getLogger(__name__)


class FileScanner:
    """Scans a folder for image files and logs each step."""

    IMAGE_PATTERNS = ['*.jpg', '*.jpeg', '*.png', '*.JPG']

    def __init__(self, folder: str):
        """Store the folder to scan."""
        self.folder = Path(folder)
        logger.debug(f'FileScanner created for {self.folder}')

    def scan(self) -> list[Path]:
        """Return a list of image files in the folder."""
        # Log the start of the scan
        logger.info(f'Scanning folder: {self.folder}')

        # Refuse to scan if the folder doesn't exist
        if not self.folder.exists():
            logger.error(f'Folder does not exist: {self.folder}')
            return []

        # Collect all images matching any pattern
        images = []
        for pattern in self.IMAGE_PATTERNS:
            found = list(self.folder.glob(pattern))
            images.extend(found)
            if found:
                logger.debug(f'  Pattern {pattern}: {len(found)} file(s)')

        # Log the final count
        logger.info(f'Found {len(images)} image(s)')
        return images
'''

# Write the file
output_path.write_text(source, encoding='utf-8')

print(f'✅ Saved: {output_path}')
print(f'   Size: {output_path.stat().st_size} bytes')

✅ Saved: C:\Users\USER\rag_course\15_pure_python\agrivoice\services\file_scanner.py
   Size: 1408 bytes


Run the Scanner on Real Images

In [15]:
import sys
from agrivoice.services.file_scanner import FileScanner

sys.path.insert(0, r'C:\Users\USER\rag_course\15_pure_python')

# Set up logging to see the scanner's messages

for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
    
logging.basicConfig(
    level=logging.DEBUG,
    format='%(levelname)-8s | %(name)-40s | %(message)s',
)

# Point it at your potato images folder
scanner = FileScanner(folder=r'C:\Users\USER\rag_course\14_multimodal_rag\data')

# Run the scan
images = scanner.scan()

print(f'\n✅ Scanner returned {len(images)} image(s)')

DEBUG    | agrivoice.services.file_scanner          | FileScanner created for C:\Users\USER\rag_course\14_multimodal_rag\data
INFO     | agrivoice.services.file_scanner          | Scanning folder: C:\Users\USER\rag_course\14_multimodal_rag\data
DEBUG    | agrivoice.services.file_scanner          |   Pattern *.jpg: 10 file(s)
DEBUG    | agrivoice.services.file_scanner          |   Pattern *.JPG: 10 file(s)
INFO     | agrivoice.services.file_scanner          | Found 20 image(s)



✅ Scanner returned 20 image(s)
